# Fiscal policy in HANK

One model folder, two versions. The variable `model_type` chooses the calibration:
- `'IKC'`: the IKC model of the lecture. Real rate rule ($r_t=r_{ss}$), real bonds and no markups ($\mu=1$). One discount factor, $B_{ss}=A_{ss}=0.25$, and the same $r_{ss}$, $G_{ss}$, $B_{ss}$ and $\kappa$ as in the RANK/TANK exercise.
- `'canonical'`: markups $\mu>1$ (stocks and capital gains), household wealth $A=5$ times annual output, $B_{ss}=1$, and two discount factors. The flags `par.taylor` (False: real rate rule) and `par.nominal_bonds` switch on the Taylor rule and nominal bonds.

In both, the government follows the debt rule $dB_t=\rho_B\left(dB_{t-1}+dG_t\right)$ (`par.rho_B`) and taxes adjust, $T_t=\tau_tY_t$. The calibration is annual.

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize

import HANKModel

## Settings

In [ ]:
plt.rcParams.update({'axes.grid': True, 'grid.color': 'black', 'grid.alpha': 0.25, 'grid.linestyle': '--', 'font.size': 12})

## Part 1: the IKC model

In [ ]:
model_type = 'IKC'

model = HANKModel.HANKModelClass(name=model_type,par={'model_type':model_type})
model.find_ss(do_print=True)
model.compute_jacs()
model.find_IRFs(shocks=['G'])

### Q1

Solve the response to the shock $dB_t=\rho_B\left(dB_{t-1}+dG_t\right)$ using the IKC:
$$
d\boldsymbol{Y}=d\boldsymbol{G}+\boldsymbol{M}\left(d\boldsymbol{Y}-d\boldsymbol{T}\right)
$$
Compare it to the solution from `model.find_IRFs()`.

*Hint:* start from $dB_{-1}=0$ and loop over $t$. The budget gives $dT_t=dG_t+(1+r_{ss})dB_{t-1}-dB_t$. Then solve $(\boldsymbol{I}-\boldsymbol{M})d\boldsymbol{Y}=d\boldsymbol{G}-\boldsymbol{M}d\boldsymbol{T}$ with `np.linalg.solve`.

In [ ]:
par = model.par
ss = model.ss
I = np.eye(par.T)
q = (1+ss.r)**(-np.arange(par.T))
M = model.jac_hh[('C_hh','Z')]
dG = model.IRF['G']

def solve_IKC(rho_B):
    """IKC solution for output and taxes under the debt rule."""

    dB = np.zeros(par.T)
    dT = np.zeros(par.T)
    for t in range(par.T):
        dB_lag = dB[t-1] if t > 0 else 0.0
        dB[t] = ...
        dT[t] = ...

    dY = ...

    return dY, dT, dB

dY_IKC, dT_IKC, dB_IKC = solve_IKC(par.rho_B)

print(f'max |dT - dT_IKC|: {np.max(np.abs(model.IRF["taxes"]-dT_IKC)):.1e}')
print(f'max |dY - dY_IKC|: {np.max(np.abs(model.IRF["Y"]-dY_IKC)):.1e}')

fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].plot(model.IRF['Y'][:30],label='find_IRFs')
axes[0].plot(dY_IKC[:30],ls='--',label='IKC')
axes[0].plot(dG[:30],ls=':',color='black',label='$dG$')
axes[0].set_title('output, $dY$')
axes[0].legend()
axes[1].plot(dG[:30],label='$dG$')
axes[1].plot(dT_IKC[:30],label='$dT$')
axes[1].plot(dB_IKC[:30],label='$dB$')
axes[1].set_title('government')
axes[1].legend()
for ax in axes: ax.set_xlabel('years')
fig.tight_layout()

### Q2

Compute the impact and cumulative multipliers as a function of $\rho_B \in \{0.8, 0.85, 0.9, 0.95, 0.99\}$. Explain your results.

*Hint:* call your function from Q1 in a loop. The cumulative multiplier is $\boldsymbol{q}^{\prime}d\boldsymbol{Y}/\boldsymbol{q}^{\prime}d\boldsymbol{G}$. Compare it with $1/\tau_{ss}$, and look at how long the boom in $d\boldsymbol{Y}$ lasts.

In [ ]:
rhos_B = [0.8,0.85,0.9,0.95,0.99]
impacts = []
cumulatives = []

fig, ax = plt.subplots(figsize=(6,4))

for rho_B in rhos_B:

    dY, dT, dB = ...
    impacts.append(...)
    cumulatives.append(...)
    ax.plot(dY[:50],label=f'$\\rho_B$ = {rho_B}')
    print(f'rho_B = {rho_B:.2f}: impact = {impacts[-1]:5.2f}, cumulative = {cumulatives[-1]:6.2f}')

print(f'1/tau = {1/ss.tau:.2f}')
ax.set_title('output, $dY$')
ax.set_xlabel('years')
ax.legend()

fig, ax = plt.subplots(figsize=(6,4))
ax.plot(rhos_B,impacts,marker='o',label='impact')
ax.plot(rhos_B,cumulatives,marker='o',label='cumulative')
ax.axhline(1/ss.tau,ls='--',color='black',label='$1/\\tau$')
ax.set_xlabel('$\\rho_B$')
ax.set_yscale('log')
ax.set_title('fiscal multipliers in HANK')
ax.legend();

## Part 2: the canonical model

In [ ]:
model_type = 'canonical'

model = HANKModel.HANKModelClass(name=model_type,par={'model_type':model_type})
model.find_ss(do_print=True)
model.compute_jacs()

### Q3

Compute the impact and cumulative multipliers as a function of $\rho_B \in \{0.8, 0.85, 0.9, 0.95, 0.99\}$ in the canonical model, with capital gains and a Taylor rule (`par.taylor = True`, real bonds). Compare to the IKC model.

*Hint:* first write `solve_multipliers`. $\rho_B$, the Taylor rule and the bond type only enter non-household blocks, so the household Jacobians do not change: recompute the others with `model.compute_jacs(skip_hh=True)`, then `model.find_IRFs(shocks=['G'])`. Then loop over $\rho_B$ with the real rate rule and with the Taylor rule. Why are the multipliers lower than in the IKC model even with the real rate rule?

In [ ]:
par = model.par
ss = model.ss
q = (1+ss.r)**(-np.arange(par.T))

def solve_multipliers(model):
    """Recompute the aggregate Jacobians, solve the G shock and return the multipliers."""

    # a. Jacobians and IRFs
    ...

    # b. multipliers
    impact = ...
    cumulative = ...

    return impact, cumulative

rhos_B = [0.8,0.85,0.9,0.95,0.99]
impacts = {}
cumulatives = {}
par.nominal_bonds = False

for taylor in [False,True]:

    par.taylor = taylor
    label = 'Taylor rule' if taylor else 'real rate rule'
    impacts[label] = []
    cumulatives[label] = []

    for rho_B in rhos_B:

        # a. set rho_B and compute the multipliers
        impact, cumulative = ...

        impacts[label].append(impact)
        cumulatives[label].append(cumulative)
        print(f'{label:15s}, rho_B = {rho_B:.2f}: impact = {impact:5.2f}, cumulative = {cumulative:5.2f}')

fig, axes = plt.subplots(1,2,figsize=(12,4))
for label in impacts:
    axes[0].plot(rhos_B,impacts[label],marker='o',label=label)
    axes[1].plot(rhos_B,cumulatives[label],marker='o',label=label)
axes[0].set_title('impact multiplier')
axes[1].set_title('cumulative multiplier')
for ax in axes:
    ax.set_xlabel('$\\rho_B$')
    ax.legend()
fig.tight_layout()

### Q4

Compute the impact and cumulative multipliers as a function of the Taylor rule coefficient, $\phi_\pi \in \{1.01, 1.25, 1.5\}$ (with $\rho_B=0.9$). What is the impact of the central bank on the multiplier?

*Hint:* print the response of the real rate on impact, `model.IRF['r'][0]`. Why does $r_t$ still rise when $\phi_\pi$ is close to 1? Look at the Taylor rule and the Fisher equation.

In [ ]:
par.taylor = True
par.nominal_bonds = False
par.rho_B = 0.9
par.kappa = 0.16

for phi_pi in [1.01,1.25,1.5]:

    # a. set phi_pi and compute the multipliers
    impact, cumulative = ...

    print(f'phi_pi = {phi_pi:.2f}: impact = {impact:5.2f}, cumulative = {cumulative:5.2f}, dr_0 = {100*model.IRF["r"][0]:.3f} pp')

### Q5

Same question with the slope of the W-NKPC, $\kappa \in \{0.08, 0.16, 0.32\}$. What is the impact?

*Hint:* print the response of inflation on impact, `model.IRF['pi'][0]`, and think about what the central bank does with it.

In [ ]:
par.taylor = True
par.nominal_bonds = False
par.rho_B = 0.9
par.phi_pi = 1.5

for kappa in [0.08,0.16,0.32]:

    # a. set kappa and compute the multipliers
    impact, cumulative = ...

    print(f'kappa = {kappa:.2f}: impact = {impact:5.2f}, cumulative = {cumulative:5.2f}, dpi_0 = {100*model.IRF["pi"][0]:.3f} pp')

### Q6

Add nominal bonds to the model (`par.nominal_bonds = True`). How does the return on household wealth $r^p_0$ (`ra`) react on impact? How does it change the multipliers?

*Hint:* on impact, $1+r^p_0=\frac{p_0+d_0}{A_{ss}}+\frac{1+i_{ss}}{1+\pi_0}\frac{B_{ss}}{A_{ss}}$. Which term moves with real bonds, and which one is added with nominal bonds? Who gains from the fall in $r^p_0$?

In [ ]:
par.taylor = True
par.rho_B = 0.9
par.phi_pi = 1.5
par.kappa = 0.16
dra = {}

for nominal_bonds in [False,True]:

    label = 'nominal bonds' if nominal_bonds else 'real bonds'

    # a. set the bond type and compute the multipliers
    impact, cumulative = ...

    # b. store the return on household wealth
    dra[label] = ...

    print(f'{label:13s}: impact = {impact:5.2f}, cumulative = {cumulative:5.2f}, dr^p_0 = {100*dra[label][0]:.3f} pp, dpi_0 = {100*model.IRF["pi"][0]:.3f} pp')

fig, ax = plt.subplots(figsize=(6,4))
for label in dra:
    ax.plot(100*dra[label][:10],marker='o',label=label)
ax.set_title('return on household wealth, $dr^p_t$ (pp)')
ax.set_xlabel('years')
ax.legend();

### Q7

Calibrate the heterogeneity in discount factors to match an average annual MPC of 0.5 in steady state, by choosing the share of patient households `par.share_H`. How do the multipliers change ($\rho_B=0.9$, real rate rule and Taylor rule)?

*Hint:* the function `calc_MPC` computes the average annual MPC from the steady-state consumption function. Each value of `share_H` needs a new steady state (`find_ss` re-solves $\beta_H$ to keep $A=5$). Find the root of `MPC_gap` with `optimize.brentq` on $[0.40,0.49]$. The household Jacobians change with the steady state, so call `model.compute_jacs()` before computing the multipliers.

In [ ]:
def calc_MPC(model):
    """Average annual MPC out of a one-time transfer in steady state."""

    par = model.par
    ss = model.ss
    dc = ss.c[:,:,1:]-ss.c[:,:,:-1]
    dm = (1+ss.ra)*(par.a_grid[1:]-par.a_grid[:-1])

    return np.sum(ss.D[:,:,:-1]*dc/dm)

print(f'baseline: share_H = {par.share_H:.3f}, MPC = {calc_MPC(model):.3f}')

def MPC_gap(share_H):
    """Gap between the annual MPC and its target of 0.5."""

    # a. set share_H and solve the steady state
    ...

    # b. gap to the target
    return ...

share_H = ...
MPC_gap(share_H)
print(f'calibrated: share_H = {share_H:.3f}, beta_H = {par.beta_H:.4f}, MPC = {calc_MPC(model):.3f}')

# c. new household Jacobians, then the multipliers
model.compute_jacs()
par.nominal_bonds = False
par.rho_B = 0.9
par.phi_pi = 1.5
par.kappa = 0.16

for taylor in [False,True]:
    par.taylor = taylor
    label = 'Taylor rule' if taylor else 'real rate rule'
    impact, cumulative = ...
    print(f'{label:15s}: impact = {impact:5.2f}, cumulative = {cumulative:5.2f}')